# Changing beta: predict, run, explain the gap

The last time you ran this model, beta stayed at 0.1185 and the two parameters that moved were the ones governing when the balancing stops. You loosened the tolerance, you tightened it, you capped the iterations, and across all of those runs one number sat still: the mean cost of a modelled trip. Tolerance is a rule about when to stop scaling, and scaling to the margins cannot change how far the model thinks people are willing to travel.

Beta can. Beta is the first parameter whose value is capable of being wrong.

Consider two versions of the model you are about to run. One has beta at 0.04, the other at 0.1778. Both converge. Both reproduce every row total and every column total to within a hundredth of a trip. Both hand you 21,025 internally consistent cells with a tidy convergence report attached. Nothing in the checking you have done so far distinguishes them, because a doubly constrained model reproduces the margins it was handed at any beta, and the margin check was designed to test exactly that and nothing else.

What separates them is elsewhere. One of them reproduces the journey lengths that people in Tyne and Wear actually recorded on Census day in 2011, and the other does not. That comparison is the only check in this notebook capable of telling you that a beta is wrong, and it is the reason the observed figures are printed on screen here when they were deliberately kept off it last time.

Before you open this notebook you should already have submitted a written prediction. If you have not, go back and do that first. What matters is not whether you predicted correctly. A learner who guesses right and cannot say why has understood less than one who guesses wrong and can explain, in terms of what beta does to the weight on a fifty-minute journey, why the model went the other way.

Work down the notebook in order.

## Where the files are

JupyterLite runs inside your browser. Nothing is installed on your machine and you do not need administrator rights, which is why this page opens on a locked-down work machine.

The notebook and its data arrived with the site, so there is nothing to download and nothing to upload. Open the file browser - the panel down the left-hand side, or the folder icon in the far-left sidebar if it is not showing - and you will find this arrangement already in place:

```
beta-sensitivity/
    changing-beta.ipynb
    data/
        zones_msoa.csv
        flows_msoa.csv
        trip_ends_msoa.csv
        cost_matrix_msoa.csv
```

Every path in the code below assumes it. The notebook sits at the top of the folder and the data sits one level under it, so moving either one will break the loading section.

**What happens to anything you change**

Because there is no server behind this, whatever you save goes into your browser's own storage rather than onto a network drive. That has two
consequences worth taking seriously. Anything you want to keep should be downloaded - right-click the file in the file browser and choose **Download**. And if you clear your browsing data, or if your employer's IT policy clears it for you, your saved work goes with it.

Do not edit the CSV files. If you want to try something out on them, duplicate one first and work on the copy.

**Getting back to the original**

Should you change the notebook and want the version you started with, use **Help > Clear Browser Data**. But read the warning it gives you before confirming. It removes everything you have stored on this site, for every notebook here, and it cannot be undone, so download anything you care about first.

## Checking the files are where you think they are

Run the cell below before anything else. It reports what it can see, which is
faster than reading an error message later and guessing what went wrong.

In [ ]:
import os

DATA_FOLDER = "data"

expected = [
    "zones_msoa.csv",
    "flows_msoa.csv",
    "trip_ends_msoa.csv",
    "cost_matrix_msoa.csv",
]

print("Looking in:", os.path.abspath(DATA_FOLDER))
print()

if not os.path.isdir(DATA_FOLDER):
    print("That folder does not exist yet.")
    print("Check the folder names and check where this notebook is saved.")
else:
    found = sorted(os.listdir(DATA_FOLDER))
    for name in expected:
        status = "found" if name in found else "MISSING"
        print(f"  {name:24s} {status}")

## Parameters

This is the only cell in the notebook you will change, and today only one of the
four values in it moves.

BETA is the sensitivity to generalised cost, per minute, in the exponential
deterrence function. It is the value you will set three times: 0.1185 to
establish a baseline, then 0.1778, then 0.04. DETERRENCE stays exponential,
because changing the shape of the function at the same time as its parameter
would leave you unable to attribute the difference to either.

MAX_ITERATIONS stays at 100 and should not be reduced. Higher betas need
substantially more iterations than the baseline did, and a run that hits the cap
has not converged, so its matrix is not comparable with one that has. TOLERANCE
stays at 0.01 trips for the same reason: three runs stopped by three different
rules would not be three points on a line.

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

BETA = 0.1185            # sensitivity to generalised cost, per minute

DETERRENCE = "exponential"   # "exponential" or "power"

MAX_ITERATIONS = 100     # most balancing iterations the model is allowed

TOLERANCE = 0.01         # a row or column total this close to its target,
                         # in trips, counts as matched

# ---------------------------------------------------------------------------

## Loading the data

The same four files as last time, read the same way. The zone list fixes the
order of everything else, so that row three of the cost matrix and row three of
the trip ends refer to the same place. What is read from each file: `zone_id`
and `zone_name` from the zone list, `origin_id`, `destination_id` and `trips`
from the flows, `origin_id`, `destination_id` and `gc_min` from the cost matrix,
and `zone_id`, `resident_workers` and `jobs` from the trip ends. The `jobs`
column lives in the trip-ends file.

The cost matrix is 21,025 rows, so give this cell a moment before deciding it
has stalled.

In [ ]:
import numpy as np
import pandas as pd

zones = pd.read_csv(f"{DATA_FOLDER}/zones_msoa.csv", encoding="utf-8-sig")
flows = pd.read_csv(f"{DATA_FOLDER}/flows_msoa.csv", encoding="utf-8-sig")
costs = pd.read_csv(f"{DATA_FOLDER}/cost_matrix_msoa.csv", encoding="utf-8-sig")
trip_ends = pd.read_csv(f"{DATA_FOLDER}/trip_ends_msoa.csv", encoding="utf-8-sig")

zone_ids = list(zones["zone_id"])
zone_names = dict(zip(zones["zone_id"], zones["zone_name"]))

print(f"Zones loaded:            {len(zone_ids)}")
print(f"Flow records:            {len(flows):,}")
print(f"Cost matrix records:     {len(costs):,}")
print(f"Trip end records:        {len(trip_ends)}")

In [ ]:
observed = (flows
            .pivot(index="origin_id", columns="destination_id", values="trips")
            .reindex(index=zone_ids, columns=zone_ids)
            .fillna(0)
            .values.astype(float))

cost = (costs
        .pivot(index="origin_id", columns="destination_id", values="gc_min")
        .reindex(index=zone_ids, columns=zone_ids)
        .values.astype(float))

margins = trip_ends.set_index("zone_id").reindex(zone_ids)
origins = margins["resident_workers"].values.astype(float)
destinations = margins["jobs"].values.astype(float)

print("Observed matrix:", observed.shape)
print("Cost matrix:    ", cost.shape)
print(f"Resident workers: {origins.sum():,.0f}     Jobs: {destinations.sum():,.0f}")

## The criterion, printed before the model runs

Here is the thing the model will be judged against, and it comes out of the
observed flows rather than out of anything the model produces. Weight every
recorded trip by the generalised cost of the pair it was made on, add them up,
divide by the number of trips, and you have the mean cost of a journey to work
as the 2011 Census recorded it across these 145 zones.

That single number is the target. The share of recorded trips costing under ten
minutes is printed beside it, because a mean can be reached by several different
distributions and the two figures together constrain the answer more tightly
than either alone.

One caveat about what these costs are. The generalised cost matrix was built on
free-flow speeds with no congestion and no junction delay, and the value of time
and vehicle operating cost behind it are stated assumptions rather than current
TAG figures. A free-flow network makes long journeys look cheaper than they are,
which matters here more than usual, since whatever the network understates the
deterrence function has to absorb.

In [ ]:
observed_trips = observed.sum()
observed_mean_cost = (observed * cost).sum() / observed_trips
observed_share_under_10 = observed[cost < 10].sum() / observed_trips * 100

print("THE OBSERVED DATA, BEFORE ANY MODEL HAS RUN")
print("-" * 56)
print(f"Recorded trips:                        {observed_trips:12,.0f}")
print(f"Mean generalised cost of a trip:       {observed_mean_cost:12.3f} minutes")
print(f"Share of trips under 10 minutes:       {observed_share_under_10:12.2f} per cent")
print()
print(f"Generalised cost, cheapest pair:       {cost.min():12.2f} minutes")
print(f"Generalised cost, dearest pair:        {cost.max():12.2f} minutes")

## The deterrence function

One line of arithmetic turns a cost into a weight, and beta is the only thing in
it that you control. Watch the two weights printed below as you change beta,
because the ratio between them is the whole of what beta does: it says how much
less attractive the dearest pair in Tyne and Wear is than the cheapest.

In [ ]:
if DETERRENCE == "exponential":
    deterrence = np.exp(-BETA * cost)
elif DETERRENCE == "power":
    deterrence = cost ** (-BETA)
else:
    raise ValueError('DETERRENCE must be "exponential" or "power"')

print(f"Deterrence function: {DETERRENCE}, beta = {BETA}")
print(f"Weight on the cheapest pair ({cost.min():.2f} min): {deterrence.max():.6f}")
print(f"Weight on the dearest pair ({cost.max():.2f} min):  {deterrence.min():.6f}")
print(f"Ratio between them:  {deterrence.max() / deterrence.min():,.0f} to 1")

## Running the model

The balancing loop is unchanged from the run you have already done, and the
iteration count it reports follows the same convention as before. Scale rows to
resident workers, scale columns to jobs, go round again until the worst
remaining gap falls under TOLERANCE.

How many times it goes round is one of the three quantities you predicted.

In [ ]:
row_factor = np.ones(len(zone_ids))
col_factor = np.ones(len(zone_ids))

history = []

for iteration in range(1, MAX_ITERATIONS + 1):

    row_factor = 1.0 / (deterrence * (col_factor * destinations)).sum(axis=1)
    col_factor = 1.0 / (deterrence * (row_factor * origins)[:, None]).sum(axis=0)

    modelled = ((row_factor * origins)[:, None]
                * (col_factor * destinations)[None, :]
                * deterrence)

    row_error = np.abs(modelled.sum(axis=1) - origins).max()
    col_error = np.abs(modelled.sum(axis=0) - destinations).max()
    worst = max(row_error, col_error)

    history.append((iteration, row_error, col_error))

    if worst < TOLERANCE:
        break

print("CONVERGENCE REPORT")
print("-" * 46)
print(f"Beta:                  {BETA}")
print(f"Iterations run:        {iteration}")
print(f"Iterations allowed:    {MAX_ITERATIONS}")
print(f"Tolerance required:    {TOLERANCE} trips")
print(f"Worst row error:       {row_error:.4f} trips")
print(f"Worst column error:    {col_error:.4f} trips")
print()
if worst < TOLERANCE:
    print("Converged.")
else:
    print("DID NOT CONVERGE within the iteration limit.")
    print("This matrix does not match its margins and is not comparable")
    print("with the other runs. Restore MAX_ITERATIONS to 100.")

## What the margins say, at every beta

The cell below is the check you made last time, reduced to two lines. Run it at
each of the three betas and watch what it does, which is nothing. Every zone's
outward trips match its resident workers and every zone's inward trips match its
jobs, at 0.04 as faithfully as at 0.1778, because that is what the two sets of
balancing factors are for.

This is worth pausing on rather than skipping. The check that reassured you at
145 zones is not a weak test of beta. It is not a test of beta at all, and a
model with a badly wrong beta will pass it every time, cheerfully, with a
verdict word attached.

In [ ]:
row_gap = modelled.sum(axis=1) - origins
col_gap = modelled.sum(axis=0) - destinations

outside = int((np.abs(row_gap) >= TOLERANCE).sum()
              + (np.abs(col_gap) >= TOLERANCE).sum())

print(f"Zone margins outside tolerance: {outside} of {2 * len(zone_ids)}")
print(f"Largest row gap:  {np.abs(row_gap).max():.4f} trips")
print(f"Largest column gap: {np.abs(col_gap).max():.4f} trips")

## This run, against the observed data

Three quantities to write down, and they are the three you predicted. Take them
from the panel below.

In [ ]:
modelled_trips = modelled.sum()
modelled_mean_cost = (modelled * cost).sum() / modelled_trips
modelled_share_under_10 = modelled[cost < 10].sum() / modelled_trips * 100

print("THIS RUN")
print("-" * 58)
print(f"Beta:                                {BETA}")
print(f"Iterations used:                     {iteration}")
print()
print(f"Modelled mean trip cost:         {modelled_mean_cost:9.3f} minutes")
print(f"Observed mean trip cost:         {observed_mean_cost:9.3f} minutes")
print(f"Modelled minus observed:         {modelled_mean_cost - observed_mean_cost:+9.3f} minutes")
print()
print(f"Modelled trips under 10 minutes: {modelled_share_under_10:9.2f} per cent")
print(f"Observed trips under 10 minutes: {observed_share_under_10:9.2f} per cent")
print(f"Modelled minus observed:         {modelled_share_under_10 - observed_share_under_10:+9.2f} points")

## Keeping the three runs beside each other

Two points give you a difference. Three give you a direction, and the direction
is what the exercise is about, so the cell below writes each run to a file
called `beta-runs.csv` in this folder and prints everything recorded so far.
Runs are keyed on beta, so running the same value twice replaces the earlier row
rather than adding a second one.

The file lives in your browser's storage like everything else here. Clear your
browsing data and it goes, in which case you re-run the three betas rather than
losing the exercise. Download it when you are finished, since the figures in it
are what your written answer has to refer to.

In [ ]:
RUN_RECORD = "beta-runs.csv"

this_run = pd.DataFrame([{
    "beta": BETA,
    "iterations": iteration,
    "mean_trip_cost_min": round(modelled_mean_cost, 3),
    "share_under_10_min_pct": round(modelled_share_under_10, 2),
}])

if os.path.exists(RUN_RECORD):
    previous = pd.read_csv(RUN_RECORD, encoding="utf-8-sig")
    keep = previous[previous["beta"].round(6) != round(BETA, 6)]
    record = pd.concat([keep, this_run], ignore_index=True)
else:
    record = this_run

record = record.sort_values("beta").reset_index(drop=True)
record.to_csv(RUN_RECORD, index=False, encoding="utf-8-sig")

print("RUNS RECORDED SO FAR")
print("-" * 58)
print(record.to_string(index=False))
print()
print(f"Observed mean trip cost:         {observed_mean_cost:.3f} minutes")
print(f"Observed share under 10 minutes: {observed_share_under_10:.2f} per cent")

## The distribution behind the mean

A mean of fourteen minutes is consistent with a great many different patterns of
travel, among them one where almost every journey takes about fourteen minutes
and one where half take four and half take twenty-four. The banded table and the
chart below open the mean up, sorting every trip into a range of generalised
cost and comparing the modelled share in each band with the observed share.

Read the two ends of the table rather than the middle. Beta governs how fast the
weight falls away as cost rises, so the bands where it shows itself most clearly
are the cheapest and the dearest.

In [ ]:
edges = [0, 5, 10, 15, 20, 25, 30, 40, 100]
labels = ["0-5", "5-10", "10-15", "15-20", "20-25", "25-30", "30-40", "40+"]

band = np.digitize(cost, edges) - 1
observed_band = np.array([observed[band == b].sum() for b in range(len(labels))])
modelled_band = np.array([modelled[band == b].sum() for b in range(len(labels))])

bands = pd.DataFrame({
    "cost_band_min": labels,
    "observed_pct": (observed_band / observed.sum() * 100).round(2),
    "modelled_pct": (modelled_band / modelled.sum() * 100).round(2),
})
bands["modelled_minus_observed"] = (bands["modelled_pct"]
                                    - bands["observed_pct"]).round(2)

print(f"TRIP COST DISTRIBUTION, beta = {BETA}")
print("-" * 58)
print(bands.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - 0.2, observed_band / observed.sum() * 100, 0.4,
       label="Observed", color="#6B7280")
ax.bar(x + 0.2, modelled_band / modelled.sum() * 100, 0.4,
       label="Modelled", color="#C8102E")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Generalised cost, minutes")
ax.set_ylabel("Share of all trips, per cent")
ax.set_title(f"Trip cost distribution, beta = {BETA}")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## What to do now

Four steps, in this order.

1. You have just run the baseline. Beta is at 0.1185, the value you used in the
   spreadsheet and in the first run at this scale. Write down the iterations
   used, the modelled mean trip cost and the share of trips under ten minutes.
2. Change BETA to 0.1778 in the parameters cell. That is the baseline increased
   by half, and it is the value your prediction was about. Restart the kernel
   and run every cell from the top, using **Kernel > Restart Kernel and Run All
   Cells**, then take the same three quantities from the panel. Open your
   submitted prediction and put it beside them.
3. Change BETA to 0.04 and restart and run all again. This is well below the
   baseline, and it converges quickly, so nothing about the run should surprise
   you except the matrix it produces.
4. Write your explanation of the gap between what you predicted and what the
   three runs did, using the recorded table and the two charts. Do that before
   reading the section below, which is why it is at the bottom of the notebook
   rather than the top.

Download `beta-runs.csv` before you leave the page.

## Read this after you have written your answer

Four things the three runs establish, the last of them only partly.

Every run converged and every run matched its margins. The verdict word printed
in the convergence report was identical at 0.04 and at 0.1778, and so was the
count of zone margins outside tolerance. On the evidence of that report alone
there is nothing to choose between the two models, which is the position a great
many people are in when they accept a modelled matrix from a supplier.

The mean trip cost moved a great deal, and it moved in the direction the
arithmetic requires. Raising beta puts more weight on cheap pairs and less on
dear ones, the balancing then distributes trips accordingly, and the average
journey gets shorter. When you loosened and tightened tolerance in the earlier
run, this same figure sat still to four decimal places, because tolerance is a
stopping rule and beta is an assumption about how people behave.

The iteration count rose sharply with beta, and this one is worth a moment. A
higher beta concentrates trips onto fewer origin-destination pairs, which makes
the row and column scalings interfere with one another more, so the balancing
takes longer to settle. Iterations are a property of the arithmetic rather than
of Tyne and Wear commuting, and a model that takes forty-three iterations rather
than thirteen is not a worse model. It is the same method working harder.

Which leaves the question the exercise turns on. Of the three betas you ran, one
lands close to the mean journey cost recorded in 2011 while the others are out
by several minutes in opposite directions, and that comparison is the first
piece of evidence you have met that can rule a parameter value out. It cannot
yet rule one in. Getting closer to the observed mean by trying values is a
search, and a search needs a rule for when to stop and a statistic for how good
the fit is, neither of which you have been given here.

That is what calibration is: estimating beta from observed travel behaviour
rather than choosing it. Beta is a parameter with a value in the world, in the
sense that the 2011 commuting pattern of Tyne and Wear constrains what it can
reasonably be, and the modeller's job is to find that value and to say how well
it is pinned down. But the estimate is only as good as the cost matrix behind
it, which here is free-flow and carries placeholder money values, so a beta
fitted to these costs is absorbing everything the network fails to represent.
Hold on to that when the calibration methods arrive.